# 04.4 — GloVe (Global Vectors)

**Problem it solves:** Word2Vec trains on local windows sequentially — it never 'sees' the global co-occurrence statistics. GloVe explicitly builds the co-occurrence matrix first, then factorizes it.

**GloVe insight:** The *ratio* of co-occurrence probabilities encodes meaning. P(ice|solid) / P(steam|solid) >> 1 because 'solid' co-occurs with 'ice' more than 'steam'. This ratio should be encoded in the dot product of word vectors.

**Objective:** wᵢ · w̃ⱼ + bᵢ + b̃ⱼ ≈ log X_{ij}, weighted by f(X_{ij})

---

In [ ]:
import numpy as np
import re
from collections import Counter, defaultdict
import random

def tokenize(text): return re.findall(r'\b[a-z]+\b', text.lower())

class GloVe:
    """
    GloVe: Global Vectors for Word Representation.
    Pennington, Socher, Manning (2014).
    
    Step 1: Build co-occurrence matrix X from corpus.
    Step 2: Minimize weighted least-squares objective:
       J = Σ f(X_ij) * (wᵢ·w̃ⱼ + bᵢ + b̃ⱼ - log X_ij)²
    """
    
    def __init__(self, embed_dim=50, window=5, x_max=100, alpha=0.75,
                 lr=0.05, n_epochs=50):
        self.embed_dim = embed_dim
        self.window = window
        self.x_max = x_max     # weighting function parameter
        self.alpha = alpha     # weighting function exponent (0.75 in paper)
        self.lr = lr
        self.n_epochs = n_epochs
    
    def _weighting_function(self, x: float) -> float:
        """
        f(x) = (x/x_max)^alpha if x < x_max, else 1.0
        Downweights very frequent co-occurrences (stopwords).
        Ensures rare pairs still contribute but don't dominate.
        """
        if x < self.x_max:
            return (x / self.x_max) ** self.alpha
        return 1.0
    
    def _build_cooccurrence(self, tokenized: list) -> dict:
        """Build sparse co-occurrence matrix as dict."""
        cooc = defaultdict(float)
        for tokens in tokenized:
            for i, w in enumerate(tokens):
                if w not in self.word2idx: continue
                for j in range(max(0, i-self.window), 
                               min(len(tokens), i+self.window+1)):
                    if i == j: continue
                    ctx = tokens[j]
                    if ctx not in self.word2idx: continue
                    dist = abs(i - j)
                    cooc[(self.word2idx[w], self.word2idx[ctx])] += 1.0/dist
        return dict(cooc)
    
    def fit(self, corpus: list[str]):
        tokenized = [tokenize(s) for s in corpus]
        counts = Counter(w for sent in tokenized for w in sent)
        self.vocab = sorted(counts.keys())
        self.word2idx = {w: i for i, w in enumerate(self.vocab)}
        V = len(self.vocab)
        
        print(f"Vocab: {V}, building co-occurrence matrix...")
        cooc = self._build_cooccurrence(tokenized)
        print(f"Non-zero co-occurrences: {len(cooc)}")
        
        # Word vectors + context vectors (GloVe keeps both)
        # Final embedding = w + w~ (average of both)
        W  = (np.random.rand(V, self.embed_dim) - 0.5) / self.embed_dim
        W_ = (np.random.rand(V, self.embed_dim) - 0.5) / self.embed_dim
        b  = np.zeros(V)
        b_ = np.zeros(V)
        
        # AdaGrad accumulators (GloVe uses AdaGrad, not SGD)
        gradsq_W  = np.ones_like(W)
        gradsq_W_ = np.ones_like(W_)
        gradsq_b  = np.ones(V)
        gradsq_b_ = np.ones(V)
        
        pairs = list(cooc.items())
        
        for epoch in range(self.n_epochs):
            random.shuffle(pairs)
            total_loss = 0.0
            
            for (i, j), x_ij in pairs:
                # Weighting
                weight = self._weighting_function(x_ij)
                log_x_ij = np.log(x_ij + 1e-10)
                
                # Prediction: dot product + biases
                pred = np.dot(W[i], W_[j]) + b[i] + b_[j]
                
                # Residual
                diff = pred - log_x_ij
                loss = weight * diff ** 2
                total_loss += loss
                
                # Gradients
                grad_common = 2 * weight * diff
                grad_wi  = grad_common * W_[j]
                grad_w_j = grad_common * W[i]
                grad_bi  = grad_common
                grad_b_j = grad_common
                
                # AdaGrad updates
                gradsq_W[i]  += grad_wi ** 2
                gradsq_W_[j] += grad_w_j ** 2
                gradsq_b[i]  += grad_bi ** 2
                gradsq_b_[j] += grad_b_j ** 2
                
                W[i]  -= self.lr * grad_wi  / np.sqrt(gradsq_W[i])
                W_[j] -= self.lr * grad_w_j / np.sqrt(gradsq_W_[j])
                b[i]  -= self.lr * grad_bi  / np.sqrt(gradsq_b[i])
                b_[j] -= self.lr * grad_b_j / np.sqrt(gradsq_b_[j])
            
            if epoch % 10 == 0:
                print(f"  Epoch {epoch}: total_loss={total_loss:.4f}")
        
        # GloVe: final embedding = average of word and context vectors
        self.vectors = (W + W_) / 2
        return self
    
    def get_vector(self, word):
        if word not in self.word2idx: return np.zeros(self.embed_dim)
        return self.vectors[self.word2idx[word]]
    
    def most_similar(self, word, n=5):
        vec = self.get_vector(word)
        norm = np.linalg.norm(vec)
        if norm == 0: return []
        norms = np.linalg.norm(self.vectors, axis=1, keepdims=True); norms[norms==0]=1
        sims = (self.vectors/norms) @ (vec/norm)
        top = sims.argsort()[::-1][1:n+1]
        return [(self.vocab[i], float(sims[i])) for i in top]

corpus = [
    "the king and queen rule the kingdom with wisdom",
    "the man and the woman walked through the park together",
    "dogs and cats are animals that make good pets",
    "the paris is the capital of france a european country",
    "london is the capital of england and a great city",
    "machine learning and deep learning are forms of artificial intelligence",
    "neural networks learn to process and understand text data",
    "ice is cold and steam is hot they are different states of water",
    "solid liquid and gas are three states of matter",
    "the king ruled the land and the queen advised the king",
] * 30

glove = GloVe(embed_dim=30, window=4, n_epochs=100, lr=0.1)
glove.fit(corpus)

In [ ]:
print("GloVe similar words:")
for word in ['king', 'france', 'learning', 'ice']:
    if word in glove.word2idx:
        print(f"  {word}: {glove.most_similar(word, 5)}")

# The weighting function: why it matters
import numpy as np
print("\nWeighting function f(x) for x_max=100, alpha=0.75:")
for x in [1, 5, 10, 50, 100, 200, 1000]:
    w = glove._weighting_function(x)
    print(f"  f({x:5d}) = {w:.4f}")

## Summary

**GloVe vs Word2Vec:**
- GloVe uses global statistics explicitly; W2V uses local windows
- GloVe often produces slightly better analogies
- GloVe training is more stable; W2V is online (can update on new data)
- In practice: very similar downstream task performance

**Why averaging W + W̃ works:** Each captures different aspects of co-occurrence; their combination is more stable than either alone.

**What breaks GloVe:**
- Polysemy (same as W2V)
- Needs the whole corpus upfront (batch, not online)
- Memory: O(V²) for the co-occurrence matrix

---
**Next:** 04.5 — FastText (subword embeddings)